# PyAEZ Module 0 — Crop Selection Utilities

Behind-the-scenes crop catalogue loading, matching, validation, and user selection functions for Module 0.

In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink
from rapidfuzz import fuzz, process

DEFAULT_CROP_METADATA_PATH = Path(
    "./data_input/metadata/PyAEZ_Parameter_Metadata.xlsx"
)

def _normalize_text(value):
    value = "" if value is None else str(value)
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower().strip()
    value = re.sub(r"[_\-]+", " ", value)
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()

def _find_column(columns, aliases):
    normalized = {
        _normalize_text(column).replace(" ", ""): column
        for column in columns
    }
    for alias in aliases:
        key = _normalize_text(alias).replace(" ", "")
        if key in normalized:
            return normalized[key]
    return None

def _standardize_crop_columns(df):
    code_col = _find_column(df.columns, ["CODE", "CROP_CODE", "Crop code"])
    name_col = _find_column(
        df.columns, ["CROP_NAME", "Crop name", "Crop", "Name"]
    )
    group_col = _find_column(
        df.columns, ["CROP_GROUP", "Crop group", "Group"]
    )

    if code_col is None or name_col is None:
        return None

    return pd.DataFrame(
        {
            "CODE": df[code_col],
            "CROP_NAME": df[name_col],
            "CROP_GROUP": (
                df[group_col]
                if group_col is not None
                else "Not specified"
            ),
        }
    )

def load_crop_catalogue(
    metadata_path=DEFAULT_CROP_METADATA_PATH,
    sheet_name=None,
):
    metadata_path = Path(metadata_path)

    if not metadata_path.exists():
        raise FileNotFoundError(
            "PyAEZ crop metadata workbook was not found:\n"
            f"{metadata_path}"
        )

    excel_file = pd.ExcelFile(metadata_path)
    candidate_sheets = (
        [sheet_name]
        if sheet_name is not None
        else excel_file.sheet_names
    )

    if sheet_name is not None and sheet_name not in excel_file.sheet_names:
        raise ValueError(
            f"Sheet {sheet_name!r} was not found. "
            f"Available sheets: {excel_file.sheet_names}"
        )

    crop_table = None
    detected_sheet = None

    for sheet in candidate_sheets:
        raw = pd.read_excel(metadata_path, sheet_name=sheet)
        standardized = _standardize_crop_columns(raw)
        if standardized is not None:
            crop_table = standardized
            detected_sheet = sheet
            break

    if crop_table is None:
        raise ValueError(
            "No valid crop catalogue was identified. "
            "The crop table must contain CODE and CROP_NAME columns."
        )

    crop_table = crop_table.dropna(
        subset=["CODE", "CROP_NAME"]
    ).copy()

    crop_table["CODE"] = crop_table["CODE"].astype(str).str.strip()
    crop_table["CROP_NAME"] = (
        crop_table["CROP_NAME"].astype(str).str.strip()
    )
    crop_table["CROP_GROUP"] = (
        crop_table["CROP_GROUP"]
        .fillna("Not specified")
        .astype(str)
        .str.strip()
    )

    crop_table = crop_table.loc[
        (crop_table["CODE"] != "")
        & (crop_table["CROP_NAME"] != "")
    ].copy()

    duplicated_codes = crop_table.loc[
        crop_table["CODE"].duplicated(keep=False), "CODE"
    ].unique()

    if len(duplicated_codes) > 0:
        raise ValueError(
            "Duplicate crop codes found: "
            f"{sorted(duplicated_codes.tolist())}"
        )

    crop_table["_SEARCH_NAME"] = crop_table["CROP_NAME"].map(
        _normalize_text
    )

    crop_table.attrs["metadata_path"] = str(metadata_path)
    crop_table.attrs["sheet_name"] = detected_sheet

    return crop_table.reset_index(drop=True)

def find_crop_matches(
    crop_name,
    crop_catalogue,
    max_options=7,
    fuzzy_cutoff=70,
):
    query = _normalize_text(crop_name)

    if not query:
        raise ValueError(
            "Please enter a crop name, for example: maize, rice or coffee."
        )

    catalogue = crop_catalogue.copy()

    exact = catalogue.loc[
        catalogue["_SEARCH_NAME"] == query
    ].copy()

    contains = catalogue.loc[
        catalogue["_SEARCH_NAME"].str.contains(
            re.escape(query), regex=True, na=False
        )
    ].copy()

    reverse_contains = catalogue.loc[
        catalogue["_SEARCH_NAME"].map(
            lambda name: bool(name) and name in query
        )
    ].copy()

    direct = pd.concat(
        [exact, contains, reverse_contains],
        ignore_index=True,
    ).drop_duplicates(subset=["CODE"], keep="first")

    fuzzy_results = process.extract(
        query,
        catalogue["_SEARCH_NAME"].tolist(),
        scorer=fuzz.WRatio,
        limit=max(max_options * 3, 15),
        score_cutoff=fuzzy_cutoff,
    )

    fuzzy = catalogue.iloc[
        [item[2] for item in fuzzy_results]
    ].copy()

    fuzzy["_FUZZY_SCORE"] = [item[1] for item in fuzzy_results]
    fuzzy["_MATCH_PRIORITY"] = 1

    direct["_MATCH_PRIORITY"] = 0
    direct["_FUZZY_SCORE"] = direct["_SEARCH_NAME"].map(
        lambda name: fuzz.WRatio(query, name)
    )

    matches = pd.concat(
        [direct, fuzzy],
        ignore_index=True,
    ).drop_duplicates(subset=["CODE"], keep="first")

    if matches.empty:
        return matches

    matches["_NAME_LENGTH"] = matches["CROP_NAME"].str.len()

    matches = matches.sort_values(
        by=[
            "_MATCH_PRIORITY",
            "_FUZZY_SCORE",
            "_NAME_LENGTH",
            "CROP_NAME",
        ],
        ascending=[True, False, True, True],
    )

    return matches.head(max_options).reset_index(drop=True)

def select_crop(
    metadata_path=DEFAULT_CROP_METADATA_PATH,
    sheet_name=None,
    max_options=7,
    default_search="",
    show_catalogue_link=True,
):
    crop_catalogue = load_crop_catalogue(
        metadata_path=metadata_path,
        sheet_name=sheet_name,
    )

    metadata_path = Path(crop_catalogue.attrs["metadata_path"])

    state = {
        "confirmed": False,
        "CROP_CODE": None,
        "CROP_NAME": None,
        "CROP_GROUP": None,
        "metadata_path": str(metadata_path),
        "sheet_name": crop_catalogue.attrs.get("sheet_name"),
    }

    search_box = widgets.Text(
        value=default_search,
        placeholder="e.g. maize, rice, coffee, wheat",
        description="Crop name:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="500px"),
    )

    search_button = widgets.Button(
        description="Find crop",
        button_style="primary",
        icon="search",
    )

    options_box = widgets.RadioButtons(
        options=[],
        description="Available options:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="750px"),
    )

    confirm_button = widgets.Button(
        description="Confirm crop",
        button_style="success",
        icon="check",
        disabled=True,
    )

    search_output = widgets.Output()
    confirmation_output = widgets.Output()

    def _run_search(_=None):
        state["confirmed"] = False
        state["CROP_CODE"] = None
        state["CROP_NAME"] = None
        state["CROP_GROUP"] = None

        confirm_button.disabled = True
        options_box.options = []

        with confirmation_output:
            clear_output()

        with search_output:
            clear_output()

            query = search_box.value.strip()

            if not query:
                print("Please enter a crop name before searching.")
                return

            matches = find_crop_matches(
                crop_name=query,
                crop_catalogue=crop_catalogue,
                max_options=max_options,
            )

            if matches.empty:
                print(
                    f'No suitable crop options were found for "{query}".'
                )
                return

            options_box.options = [
                (
                    f"{row['CROP_NAME']} [{row['CROP_GROUP']}]",
                    row["CODE"],
                )
                for row in matches.to_dict("records")
            ]

            if options_box.options:
                options_box.value = options_box.options[0][1]
                confirm_button.disabled = False

            print(
                f'Closest available PyAEZ crop options for "{query}":'
            )

    def _confirm_crop(_=None):
        selected_code = options_box.value

        if not selected_code:
            return

        selected = crop_catalogue.loc[
            crop_catalogue["CODE"] == selected_code
        ].iloc[0]

        state["confirmed"] = True
        state["CROP_CODE"] = selected["CODE"]
        state["CROP_NAME"] = selected["CROP_NAME"]
        state["CROP_GROUP"] = selected["CROP_GROUP"]

        with confirmation_output:
            clear_output()
            print("Crop confirmed")
            print("-" * 45)
            print(f"Crop name : {state['CROP_NAME']}")
            print(f"Crop group: {state['CROP_GROUP']}")
            print("-" * 45)

    search_button.on_click(_run_search)
    confirm_button.on_click(_confirm_crop)

    print(
        "Enter the crop you want to evaluate. "
        "You do not need to know the PyAEZ crop code."
    )

    if show_catalogue_link:
        print(
            "For reference, the complete crop catalogue is available here:"
        )
        display(FileLink(str(metadata_path)))

    display(
        widgets.VBox(
            [
                widgets.HBox([search_box, search_button]),
                search_output,
                options_box,
                confirm_button,
                confirmation_output,
            ]
        )
    )

    if default_search.strip():
        _run_search()

    return state

def get_confirmed_crop(crop_selection):
    if not isinstance(crop_selection, dict):
        raise TypeError(
            "crop_selection must be the state returned by select_crop()."
        )

    if not crop_selection.get("confirmed", False):
        raise ValueError(
            "No crop has been confirmed. "
            "Please search for a crop and click 'Confirm crop' before continuing."
        )

    return (
        crop_selection["CROP_CODE"],
        crop_selection["CROP_NAME"],
        crop_selection["CROP_GROUP"],
    )